In [4]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage
from langchain_core.prompts import ChatPromptTemplate
import os
import json
from typing import Dict, Literal
from pydantic import BaseModel, Field

In [8]:
class Step(BaseModel):
    """Step."""
    phase: Literal["plan", "solve", "critique", "finalize"] = Field(
        description="The current reasoning phase."
    )
    content: str = Field(
        description="The reasoning or answer for this phase."
    )


def execute_agent(llm, n_max_steps: int):
    """Executes agent."""
    
    messages = [
        ("system", system_prompt),
        ("human", prompt),
    ]

    for ix in range(n_max_steps):
        # call llm
        response: Step = llm.invoke(messages)

        print(f"Step={ix} phase={response.phase} content={response.content}")

        # if done, return answer
        if response.phase == 'finalize':
            return response.content

        # execute next phase
        messages.append( ("ai", response.model_dump_json()) )
        messages.append( ("human", "continue with next phase") )

    return f"Reached {n_max_steps}"
    
    
system_prompt = """
You are an agentic Chain of thought reasoner.
Work in four phases: plan -> solve -> critique -> finalize.
Always output the most appropriate next step {{"phase": ... , "content": ...}}.
"""

prompt = """
Mary has two cows, I have two sheeps, how many animals do we have?
"""

model = "gpt-4.1-mini"

llm = ChatOpenAI(
    model=model
    temperature=0,
    api_key=os.getenv("OPENAI_API_KEY"),
).with_structured_output(Step)

n_max_steps = 5
execute_agent(llm=llm, n_max_steps=n_max_steps)

Step=0 phase=plan content=I need to determine the total number of animals Mary and I have together. Mary has two cows, and I have two sheep. The total number of animals is the sum of these two quantities.
Step=1 phase=solve content=Mary has 2 cows and I have 2 sheep. Adding these together, 2 + 2 = 4. Therefore, we have a total of 4 animals.
Step=2 phase=critique content=The calculation is straightforward and correct. Adding the number of cows and sheep gives the total number of animals. There are no ambiguities or errors in the reasoning.
Step=3 phase=finalize content=Mary and I have a total of 4 animals: 2 cows and 2 sheep combined.


'Mary and I have a total of 4 animals: 2 cows and 2 sheep combined.'